# Preregistered Target Trial Emulation: SGLT2i vs. DPP-4i on 3-Point MACE
**Protocol ID:** PREREG-EHR-SGLT2-2026-001  
**Objective:** Evaluate 2-year comparative effectiveness of SGLT2 inhibitors versus DPP-4 inhibitors on incident Major Adverse Cardiovascular Events (3-point MACE) using an active comparator, new-user cohort design with 1:1 propensity score caliper matching.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import ALL_COVARIATES, SMD_ACCEPTANCE_THRESHOLD
from src.synthetic_generator import generate_synthetic_raw_cohort
from src.cohort_definition import apply_cohort_eligibility
from src.statistics import (
    estimate_propensity_scores,
    trim_propensity_scores,
    perform_nearest_neighbor_matching,
    calculate_smd,
    fit_cox_ph_model,
    calculate_absolute_risk_and_nnt,
)
print("All analysis modules successfully imported.")

## 1. Raw Fixture Ingestion & Schema Inspection

In [ ]:
df_raw = generate_synthetic_raw_cohort(n_patients=6000, random_seed=42)
print(f"Raw dataset dimensions: {df_raw.shape}")
df_raw[['patient_id', 'age', 'female', 'index_drug', 'baseline_hba1c', 'baseline_egfr', 'baseline_ascvd', 'mace_event']].head(8)

## 2. Monotonic Cohort Exclusion Cascade & Attrition Waterfall

In [ ]:
df_eligible, df_attrition = apply_cohort_eligibility(df_raw, save_interim=False)
print(f"Eligible cohort retained: {len(df_eligible)} patients")
df_attrition

## 3. Propensity Score Modeling & Covariate Balance (Love Plot)

In [ ]:
# Estimate Propensity Scores & Trim Common Support
df_ps = estimate_propensity_scores(df_eligible, seed=42)
df_trimmed = trim_propensity_scores(df_ps)

# 1:1 Nearest-Neighbor Caliper Matching
df_matched, caliper_width = perform_nearest_neighbor_matching(df_trimmed, seed=42)

smd_pre = calculate_smd(df_trimmed, treatment_col="treatment")
smd_post = calculate_smd(df_matched, treatment_col="treatment")
balance_df = pd.merge(smd_pre, smd_post, on="covariate", suffixes=("_unmatched", "_matched"))
balance_df['balance_achieved (<0.10)'] = balance_df['smd_matched'] < SMD_ACCEPTANCE_THRESHOLD
print(f"Caliper width: {caliper_width:.4f}")
balance_df

## 4. Cox Proportional Hazards Model & Causal Effect Estimation

In [ ]:
# Primary Outcome: On-treatment 3-point MACE
primary_res = fit_cox_ph_model(
    df_matched,
    duration_col="fu_days_primary",
    event_col="mace_event",
    treatment_col="treatment",
    cluster_col="pair_id",
    analysis_name="Primary 3-Point MACE (Matched As-Treated)",
)

# Negative Control Outcome: Accidental Trauma / Fracture
neg_res = fit_cox_ph_model(
    df_matched,
    duration_col="fu_days_neg_control",
    event_col="neg_control_event",
    treatment_col="treatment",
    cluster_col="pair_id",
    analysis_name="Negative Control (Trauma/Fracture)",
)

abs_metrics = calculate_absolute_risk_and_nnt(df_matched)

print("=== PRIMARY ANALYSIS RESULTS ===")
print(f"Hazard Ratio: {primary_res.hazard_ratio:.3f} (95% CI: {primary_res.ci_lower:.3f} - {primary_res.ci_upper:.3f})")
print(f"P-value: {primary_res.p_value:.4f}")
print(f"E-value (Point Estimate): {primary_res.e_value_estimate:.2f}")
print(f"E-value (CI Bound): {primary_res.e_value_ci_bound:.2f}")
print(f"Schoenfeld PH Test P-value: {primary_res.schoenfeld_ph_pvalue:.3f}")
print(f"Absolute Risk Reduction (2-yr): {abs_metrics['absolute_risk_reduction']*100:.2f}%")
print(f"Number Needed to Treat (NNT): {abs_metrics['number_needed_to_treat']}")
print("\n=== NEGATIVE CONTROL OUTCOME ===")
print(f"Hazard Ratio: {neg_res.hazard_ratio:.3f} (95% CI: {neg_res.ci_lower:.3f} - {neg_res.ci_upper:.3f})")
print(f"P-value: {neg_res.p_value:.4f}")